In [1]:
!pip install pyspark==3.5.1

In [2]:
!apt update -qq
!apt install -y openjdk-11-jdk-headless

66 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openjdk-11-jdk-headless is already the newest version (11.0.30+7-1ubuntu1~22.04).
0 upgraded, 0 newly installed, 0 to remove and 66 not upgraded.


In [3]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [4]:
from pyspark import SparkConf, SparkContext
from pyspark.sql import SQLContext

In [5]:
from pyspark.sql import SparkSession

# Create Spark Session
spark = SparkSession.builder \
    .appName("films") \
    .master("local[*]") \
    .getOrCreate()

# Spark Context
sc = spark.sparkContext

In [7]:
house_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/content/Boston.csv")

house_df.show()

+---+-------+----+-----+----+-----+-----+-----+------+---+---+-------+------+-----+----+
|_c0|   crim|  zn|indus|chas|  nox|   rm|  age|   dis|rad|tax|ptratio| black|lstat|medv|
+---+-------+----+-----+----+-----+-----+-----+------+---+---+-------+------+-----+----+
|  1|0.00632|18.0| 2.31|   0|0.538|6.575| 65.2|  4.09|  1|296|   15.3| 396.9| 4.98|24.0|
|  2|0.02731| 0.0| 7.07|   0|0.469|6.421| 78.9|4.9671|  2|242|   17.8| 396.9| 9.14|21.6|
|  3|0.02729| 0.0| 7.07|   0|0.469|7.185| 61.1|4.9671|  2|242|   17.8|392.83| 4.03|34.7|
|  4|0.03237| 0.0| 2.18|   0|0.458|6.998| 45.8|6.0622|  3|222|   18.7|394.63| 2.94|33.4|
|  5|0.06905| 0.0| 2.18|   0|0.458|7.147| 54.2|6.0622|  3|222|   18.7| 396.9| 5.33|36.2|
|  6|0.02985| 0.0| 2.18|   0|0.458| 6.43| 58.7|6.0622|  3|222|   18.7|394.12| 5.21|28.7|
|  7|0.08829|12.5| 7.87|   0|0.524|6.012| 66.6|5.5605|  5|311|   15.2| 395.6|12.43|22.9|
|  8|0.14455|12.5| 7.87|   0|0.524|6.172| 96.1|5.9505|  5|311|   15.2| 396.9|19.15|27.1|
|  9|0.21124|12.5| 7.

In [8]:
## Printing schema
house_df.printSchema()


root
 |-- _c0: integer (nullable = true)
 |-- crim: double (nullable = true)
 |-- zn: double (nullable = true)
 |-- indus: double (nullable = true)
 |-- chas: integer (nullable = true)
 |-- nox: double (nullable = true)
 |-- rm: double (nullable = true)
 |-- age: double (nullable = true)
 |-- dis: double (nullable = true)
 |-- rad: integer (nullable = true)
 |-- tax: integer (nullable = true)
 |-- ptratio: double (nullable = true)
 |-- black: double (nullable = true)
 |-- lstat: double (nullable = true)
 |-- medv: double (nullable = true)



In [9]:
## Descriptive analysis
house_df.toPandas()

,_c0,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,black,lstat,medv
0,1,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,2,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,3,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,4,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,5,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,502,0.06263,0.0,11.93,0,0.573,6.593,69.1,2.4786,1,273,21.0,391.99,9.67,22.4
502,503,0.04527,0.0,11.93,0,0.573,6.120,76.7,2.2875,1,273,21.0,396.90,9.08,20.6
503,504,0.06076,0.0,11.93,0,0.573,6.976,91.0,2.1675,1,273,21.0,396.90,5.64,23.9
504,505,0.10959,0.0,11.93,0,0.573,6.794,89.3,2.3889,1,273,21.0,393.45,6.48,22.0


In [10]:
from pyspark.ml.feature import VectorAssembler
vectorAssembler = VectorAssembler(inputCols = ['crim', 'zn', 'indus', 'chas', 'nox', 'rm', 'age', 'dis', 'rad', 'tax', 'ptratio', 'black', 'lstat'], outputCol = 'features')
vhouse_df = vectorAssembler.transform(house_df)
vhouse_df = vhouse_df.select(['features', 'medv'])
vhouse_df.show(3)

+--------------------+----+
|            features|medv|
+--------------------+----+
|[0.00632,18.0,2.3...|24.0|
|[0.02731,0.0,7.07...|21.6|
|[0.02729,0.0,7.07...|34.7|
+--------------------+----+
only showing top 3 rows



In [11]:
splits = vhouse_df.randomSplit([0.7, 0.3])
train_df = splits[0]
test_df = splits[1]
#train_df,test_df=vhouse_df.randomSplit([0.7,0.3])

In [12]:
from pyspark.ml.regression import LinearRegression
lr = LinearRegression(featuresCol = 'features', labelCol='medv', maxIter=10)
lr_model = lr.fit(train_df)
print("Coefficients: " + str(lr_model.coefficients))
print("Intercept: " + str(lr_model.intercept))

Coefficients: [-0.13981504782859575,0.052276500660515306,-0.018503863453529994,3.4616478293918806,-13.670904358217365,3.733976213334767,0.0010823295590518733,-1.5714521885000599,0.338560368728075,-0.013512745534964886,-0.9253757794377393,0.009755952083273914,-0.5399532298426646]
Intercept: 35.28916990091892


In [13]:
trainingSummary = lr_model.summary
print("RMSE: %f" % trainingSummary.rootMeanSquaredError)
print("r2: %f" % trainingSummary.r2)

RMSE: 4.788179
r2: 0.740394


In [14]:
lr_predictions = lr_model.transform(test_df)
lr_predictions.select("prediction","medv","features").show(5)
from pyspark.ml.evaluation import RegressionEvaluator
lr_evaluator = RegressionEvaluator(predictionCol="prediction", \
                 labelCol="medv",metricName="r2")
print("R Squared (R2) on test data = %g" % lr_evaluator.evaluate(lr_predictions))

+------------------+----+--------------------+
|        prediction|medv|            features|
+------------------+----+--------------------+
|31.416748314117292|32.2|[0.00906,90.0,2.9...|
|30.122057431085402|32.7|[0.01301,35.0,1.5...|
|41.080740412719656|50.0|[0.01381,80.0,0.4...|
|25.209780008508638|30.1|[0.01709,90.0,2.0...|
|19.777387047764805|20.1|[0.01965,80.0,1.7...|
+------------------+----+--------------------+
only showing top 5 rows

R Squared (R2) on test data = 0.724323


In [15]:
print("numIterations: %d" % trainingSummary.totalIterations)
print("objectiveHistory: %s" % str(trainingSummary.objectiveHistory))
trainingSummary.residuals.show()


numIterations: 0
objectiveHistory: [0.0]
+--------------------+
|           residuals|
+--------------------+
|  -6.389503413584354|
|  -4.983573424308531|
|  4.1745277257990985|
|  3.8028639435846756|
| -1.6449050957249227|
|  -2.325114357217643|
|  -3.098992006419717|
|   4.457330860416221|
|   6.455668760962894|
|   2.355021436483703|
| -2.0776696574753686|
|  10.040752855065705|
|   6.515341866509587|
|-0.04100846631442...|
|  5.2827892605635824|
| -0.8579572672283824|
|  -5.362404847481425|
|   4.276166262377565|
|  -3.270884300967758|
| -0.8651502316513593|
+--------------------+
only showing top 20 rows



In [16]:
predictions = lr_model.transform(test_df)
predictions.select("prediction","medv","features").show()

+------------------+----+--------------------+
|        prediction|medv|            features|
+------------------+----+--------------------+
|31.416748314117292|32.2|[0.00906,90.0,2.9...|
|30.122057431085402|32.7|[0.01301,35.0,1.5...|
|41.080740412719656|50.0|[0.01381,80.0,0.4...|
|25.209780008508638|30.1|[0.01709,90.0,2.0...|
|19.777387047764805|20.1|[0.01965,80.0,1.7...|
|27.931932118448604|23.9|[0.02543,55.0,3.7...|
|19.524157344433533|18.5|[0.03041,0.0,5.19...|
|28.713455752523927|31.2|[0.03049,55.0,3.7...|
|30.010669509275758|34.9|[0.0315,95.0,1.47...|
|  22.4934643387529|20.6|[0.03306,0.0,5.19...|
|34.523091750443605|34.9|[0.03359,75.0,2.9...|
|29.284904172937292|24.1|[0.03445,82.5,2.0...|
|21.278909365635762|20.9|[0.03548,80.0,3.6...|
|24.999454351671446|22.9|[0.03551,25.0,4.8...|
|30.331672710963428|23.5|[0.03584,80.0,3.3...|
| 34.36401343724991|35.4|[0.03705,20.0,3.3...|
|21.729025137537192|20.7|[0.03738,0.0,5.19...|
|26.840383549277025|23.2|[0.03871,52.5,5.3...|
|26.634647872

## Decision tree regression

In [17]:
from pyspark.ml.regression import DecisionTreeRegressor
dt = DecisionTreeRegressor(featuresCol ='features', labelCol = 'medv')
dt_model = dt.fit(train_df)
dt_predictions = dt_model.transform(test_df)
dt_evaluator = RegressionEvaluator(
    labelCol="medv", predictionCol="prediction", metricName="rmse")
rmse = dt_evaluator.evaluate(dt_predictions)
print("Root Mean Squared Error (RMSE) on test data = %g" % rmse)

Root Mean Squared Error (RMSE) on test data = 3.42516


In [19]:
from pyspark.ml.regression import DecisionTreeRegressor
dt = DecisionTreeRegressor(featuresCol ='features', labelCol = 'medv')
dt_model = dt.fit(train_df)
dt_predictions = dt_model.transform(test_df)
dt_evaluator = RegressionEvaluator(
    labelCol="medv", predictionCol="prediction", metricName="r2")
r2 = dt_evaluator.evaluate(dt_predictions)
print("R2 on test data = %g" % r2)

R2 on test data = 0.840853


In [20]:
lr_evaluator.evaluate(dt_predictions)

0.840852777588143

In [21]:
 dt_model.featureImportances

SparseVector(13, {0: 0.0292, 2: 0.0023, 4: 0.0056, 5: 0.2804, 6: 0.0085, 7: 0.0752, 10: 0.0307, 11: 0.0033, 12: 0.5648})

In [22]:
house_df.take(1)

[Row(_c0=1, crim=0.00632, zn=18.0, indus=2.31, chas=0, nox=0.538, rm=6.575, age=65.2, dis=4.09, rad=1, tax=296, ptratio=15.3, black=396.9, lstat=4.98, medv=24.0)]

## Gradient-boosted tree regression

In [23]:
from pyspark.ml.regression import GBTRegressor
gbt = GBTRegressor(featuresCol = 'features', labelCol = 'medv', maxIter=10)
gbt_model = gbt.fit(train_df)
gbt_predictions = gbt_model.transform(test_df)
gbt_predictions.select('prediction', 'medv', 'features').show(5)

+------------------+----+--------------------+
|        prediction|medv|            features|
+------------------+----+--------------------+
|36.400388108712704|32.2|[0.00906,90.0,2.9...|
| 33.54897052001265|32.7|[0.01301,35.0,1.5...|
|  49.0620998687659|50.0|[0.01381,80.0,0.4...|
| 29.91447975638859|30.1|[0.01709,90.0,2.0...|
|19.225314959196627|20.1|[0.01965,80.0,1.7...|
+------------------+----+--------------------+
only showing top 5 rows



In [24]:
gbt_evaluator = RegressionEvaluator(
    labelCol="medv", predictionCol="prediction", metricName="rmse")
rmse = gbt_evaluator.evaluate(gbt_predictions)
print("Root Mean Squared Error (RMSE) on test data = %g" % rmse)

Root Mean Squared Error (RMSE) on test data = 3.30669


In [25]:
gbt_evaluator = RegressionEvaluator(
    labelCol="medv", predictionCol="prediction", metricName="r2")
r2 = gbt_evaluator.evaluate(gbt_predictions)
print("R2 Score on test data = %g" % r2)

R2 Score on test data = 0.851671
